In [ ]:
!git clone https://github.com/Mr-Gump/seq-model.git

In [1]:
import os
import sys

IS_COLAB = 'COLAB_RELEASE_TAG' in os.environ

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATASET_DIR = '/content/drive/dataset/seq-data/'
    sys.path.append("/content/seq-data")
else:
    DATASET_DIR = './dataset'

In [ ]:
import pandas as pd
import os

seq_data_path = os.path.join(DATASET_DIR, 'event_lst.csv')
sample_path = os.path.join(DATASET_DIR, 'tmp_0403.csv')

df = pd.read_csv(seq_data_path)
df['event_lst'] = df['event_lst'].apply(eval)
df['event_num'] = df['event_lst'].apply(len)

In [ ]:
df_sample = pd.read_csv(sample_path)

In [ ]:
df.drop_duplicates(subset=['sn'], inplace=True)
df_sample.drop_duplicates(subset=['sn'], inplace=True)

In [ ]:
def get_event_seq(event_lst):
    return [event['event_id'] for event in event_lst]

def get_time_delta_seq(event_lst):
    return [event['time_delta'] for event in event_lst]

def get_value_seq(event_lst):
    return [
        (event['onway_amt'], event['onway_cnt'], event['borrow_amt'], event['is_same_pkg'])
        for event in event_lst
    ]

In [ ]:
df['event_seq'] = df['event_lst'].apply(get_event_seq)
df['time_delta_seq'] = df['event_lst'].apply(get_time_delta_seq)
df['value_seq'] = df['event_lst'].apply(get_value_seq)

## 针对 log1p 压缩特性，将时间差序列转换为剩余时间序列，以更好地捕捉时间信息

In [ ]:
import numpy as np

def handle_time_delta_seq(time_delta_seq):
    sum = 0
    total_sum = np.sum(time_delta_seq)
    for i in range(len(time_delta_seq)):
        sum += time_delta_seq[i]
        time_delta_seq[i] = total_sum - sum
    return time_delta_seq

df['time_delta_seq'] = df['time_delta_seq'].apply(handle_time_delta_seq)

In [ ]:
df_merge = pd.merge(df_sample, df[['sn', 'event_seq','time_delta_seq','value_seq']], on='sn', how='inner')

In [ ]:
df_merge

In [ ]:
from sklearn.model_selection import train_test_split

df_merge = df_merge.sort_values('verify_time').reset_index(drop=True)

n = len(df_merge)
oot_start = int(n * 0.8)

df_in_time = df_merge.iloc[:oot_start]
df_oot = df_merge.iloc[oot_start:]

# 同一时间段内随机划分训练集(5/8)和测试集(3/8)，即总体 50%:30%
df_train, df_test = train_test_split(df_in_time, test_size=3/8, random_state=42, stratify=df_in_time['target'])

print(f"训练集: {len(df_train)} ({len(df_train)/n:.1%}), 日期范围: {df_train['verify_time'].min()} ~ {df_train['verify_time'].max()}")
print(f"测试集: {len(df_test)} ({len(df_test)/n:.1%}), 日期范围: {df_test['verify_time'].min()} ~ {df_test['verify_time'].max()}")
print(f"验证集(OOT): {len(df_oot)} ({len(df_oot)/n:.1%}), 日期范围: {df_oot['verify_time'].min()} ~ {df_oot['verify_time'].max()}")
print(f"\ntarget分布:")
print(f"  训练集: {df_train['target'].mean():.4f}")
print(f"  测试集: {df_test['target'].mean():.4f}")
print(f"  验证集: {df_oot['target'].mean():.4f}")

In [ ]:
from event_classifier_v2.main import main
import warnings

warnings.filterwarnings("ignore")

In [ ]:
df_out = main(df_train, df_test, df_oot)

In [ ]:
df_test_res,df_oot_res = df_out[1],df_out[2]

In [ ]:
def prob_to_score(prob):
    factor = 20 / np.log(2)
    offset = 600 + factor * np.log(0.284)
    odds = prob / (1 - prob)
    score = offset - factor * np.log(odds)
    return score

In [ ]:
df_test_res['score'] = df_test_res['prob'].apply(prob_to_score)
df_oot_res['score'] = df_oot_res['prob'].apply(prob_to_score)

In [ ]:
import toad

bucket = toad.metrics.KS_bucket(df_oot_res['score'], df_oot_res['target'], bucket=10, method='quantile')
bucket

In [ ]:
bucket = toad.metrics.KS_bucket(df_oot_res['reloan_score_2508_score'], df_oot_res['target'], bucket=10, method='quantile')
bucket

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.heatmap(df_test_res[['score', 'reloan_score_2508_score']].corr(), annot=True, cmap='RdBu_r', vmin=-1, vmax=1)
plt.show()

In [ ]:
# 1. 分别等频十分箱
df_oot_res['score_bin'] = pd.qcut(df_oot_res['score'], q=10, duplicates='drop')
df_oot_res['reloan_score_2508_score_bin'] = pd.qcut(df_oot_res['reloan_score_2508_score'], q=10, duplicates='drop')

# 2. 交叉统计 target 浓度（坏样本率）
cross = df_oot_res.pivot_table(
    values='target',
    index='score_bin',
    columns='reloan_score_2508_score_bin',
    aggfunc='mean'
)

# 3. 交叉统计每个格子的样本量（辅助参考）
cross_count = df_oot_res.pivot_table(
    values='target',
    index='score_bin',
    columns='reloan_score_2508_score_bin',
    aggfunc='count'
)

# 4. 热力图可视化
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

sns.heatmap(cross, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[0])
axes[0].set_title('bad_rate')
axes[0].set_xlabel('reloan_score_2508_score_bin')
axes[0].set_ylabel('score_bin')

sns.heatmap(cross_count, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title('loan_cnt')
axes[1].set_xlabel('reloan_score_2508_score_bin')
axes[1].set_ylabel('score_bin')

plt.tight_layout()
plt.show()